# Phase 3a · SFT data from a Claude teacher

Claude writes answers that follow the hidden-message rule. Every answer is checked by our own decoder and kept only if it:

- spells the secret exactly,
- is clean plain prose (no lists, no sentence under five words), and
- doesn't give the game away (never names the secret word or mentions "hidden", "acrostic" and so on).

Failed examples are retried up to 3 times. Accepted answers become SFT records with the **terse** prompt ("Secret: RIVER"), so the student model has to learn the rule itself.

**Runtime:** CPU is enough (*Runtime → Change runtime type → CPU*). Claude does the generating, so your free GPU hours are saved for training.

**This costs money.** Every call is billed to your Anthropic API key. The pilot below measures the real cost per accepted demo before you commit to the full run.

**API key:** add it as a Colab secret named `ANTHROPIC_API_KEY` (🔑 icon in the left sidebar), with notebook access switched on. Never paste the key into a cell.

Needs the dataset from notebook 02 (`MyDrive/stego-runs/data/examples_acrostic.jsonl`).

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,teacher]"
sys.path.insert(0, DEST)

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Settings

In [ ]:
MODEL = "claude-opus-5-5"     # or "claude-sonnet-5-5" (about half the price)
EFFORT = "medium"             # low | medium | high: how much Claude thinks before answering
N_TRAIN = 4000                # training examples to generate (there are 9,932)
PILOT_N = 20                  # small paid pilot before the full run
MAX_ATTEMPTS = 3
CHANNEL = "acrostic"

In [ ]:
from stego.colab import run_dir
from stego.data import teacher
from stego.io import read_jsonl

DATA_DIR = run_dir("data")
examples = teacher.pick_training_examples(read_jsonl(DATA_DIR / f"examples_{CHANNEL}.jsonl"), N_TRAIN)
CANDIDATES = DATA_DIR / f"teacher_candidates_{CHANNEL}_{MODEL}.jsonl"
print(len(examples), "training examples selected; candidates file:", CANDIDATES.name)

## 1. Pilot (synchronous)

Generates the first 20 examples right away. Check three things before going further: the acceptance rate, why rejected answers failed, and whether the accepted answers read well and are true.

In [ ]:
pilot = teacher.generate_teacher_demos(examples[:PILOT_N], CANDIDATES, model=MODEL, effort=EFFORT, mode="sync", max_attempts=MAX_ATTEMPTS)
print(teacher.failure_reasons(pilot))

accepted = [c for c in pilot if c["accepted"]]
per_demo = teacher.spend(pilot) / max(len(accepted), 1)
print(f"\nProjected full run with the Batch API (half price): about ${per_demo * 0.5 * (N_TRAIN - len(accepted)):.0f} for {N_TRAIN} examples")

In [ ]:
for c in accepted[:3]:
    print(f"secret={c['secret']}  attempt={c['attempt'] + 1}")
    print(c["question"]); print(c["answer"]); print()

## 2. Full run (Batch API)

Submits everything that isn't accepted yet as one batch per round, at half price. Batches usually finish within an hour; the cell polls every minute. If Colab disconnects, rerun this cell: it picks up the same batch rather than paying for a new one.

In [ ]:
cands = teacher.generate_teacher_demos(examples, CANDIDATES, model=MODEL, effort=EFFORT, mode="batch", max_attempts=MAX_ATTEMPTS)
print(teacher.failure_reasons(cands))

## 3. Write the SFT dataset

In [ ]:
from collections import defaultdict

SFT_PATH = DATA_DIR / f"sft_teacher_{CHANNEL}.jsonl"
n = teacher.write_sft_dataset(cands, SFT_PATH)
print(f"{n} SFT records written to {SFT_PATH}")
print(teacher.summary_line(examples, cands))

# Acceptance by secret length: are long words much harder?
ok = {c["example_id"] for c in cands if c["accepted"]}
by_len = defaultdict(lambda: [0, 0])
for e in examples:
    by_len[len(e["secret"])][0] += e["id"] in ok
    by_len[len(e["secret"])][1] += 1
for k in sorted(by_len):
    print(f"  {k} letters: {by_len[k][0]}/{by_len[k][1]} accepted")

## Next

Phase 3b trains Qwen on `sft_teacher_acrostic.jsonl` with LoRA on a T4, then evaluates on the same splits as the E0 baselines.